In [ ]:
!git clone https://github.com/sonlam1102/vihsd.git
!pip install -q transformers datasets evaluate accelerate scikit-learn pyvi

Cloning into 'vihsd'...
remote: Enumerating objects: 66, done.
remote: Counting objects: 100% (66/66), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 66 (delta 26), reused 15 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (66/66), 1.87 MiB | 5.19 MiB/s, done.
Resolving deltas: 100% (26/26), done.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

train_df = pd.read_csv("/content/drive/MyDrive/ToxicDataset/train3.csv")
dev_df = pd.read_csv("/content/drive/MyDrive/ToxicDataset/dev.csv")

print("Train shape:", train_df.shape)
print(train_df.head())

Train shape: (2095, 2)
                                           free_text  label_id
0  Hứng rồi hứng rồi. M shot rồi. Hứng rồi. Hứng ...         0
1                                          [âm nhạc]         0
2                                        nó cầm sao?         0
3  Ờ thằng de block bên này. Block bên bck tường ...         0
4  Em chào đại ca. Ơ Ry đẩy rồi em ơi. Ry đẩy qua...         1


In [ ]:
from datasets import Dataset

# skip unreadable rows
train_df = train_df.dropna(subset=["free_text"])
dev_df = dev_df.dropna(subset=["free_text"])

# Rename the column 'label_id' to 'label' as requested
train_df = train_df.rename(columns={"label_id": "label"})
dev_df = dev_df.rename(columns={"label_id": "label"})

label_mapping = {"CLEAN": 0, "OFFENSIVE": 1, "HATE": 2}
if "label" in train_df.columns and train_df["label"].dtype == object:
    train_df["label"] = train_df["label"].map(label_mapping)
    dev_df["label"] = dev_df["label"].map(label_mapping)

train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(dev_df)

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "vinai/phobert-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(examples["free_text"], padding="max_length", truncation=True, max_length=256)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)



config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

Map:   0%|          | 0/2095 [00:00<?, ? examples/s]

Map:   0%|          | 0/2672 [00:00<?, ? examples/s]

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [ ]:
import numpy as np
import evaluate
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments


model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint, num_labels=3
)

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
)


trainer.train()

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  540MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors: reconstructing file:   0%|          |  0.00B /  540MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.496009,0.829716
2,No log,0.508323,0.836452
3,No log,0.548826,0.821482


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=393, training_loss=0.4772991539564448, metrics={'train_runtime': 488.9061, 'train_samples_per_second': 12.855, 'train_steps_per_second': 0.804, 'total_flos': 826833915210240.0, 'train_loss': 0.4772991539564448, 'epoch': 3.0})

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


save_directory = "/content/drive/MyDrive/vihsd_toxicity_model"
trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)

print("debug")

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

save_directory = "/content/drive/MyDrive/vihsd_toxicity_model"

# 1. Explicitly set num_labels to match your classes (e.g., 3 for ViHSD)
model = AutoModelForSequenceClassification.from_pretrained(
    save_directory,
    num_labels=3
)
tokenizer = AutoTokenizer.from_pretrained(save_directory)

# 2. (Optional) If it still acts up, force the pipeline onto the CPU to see a readable error:
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=-1)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
from transformers import pipeline

classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=-1)

texts = [
"hôm nay đen vãi lồn",
"cay vcl",
"chào cậu",
"ăn ba tô cơm",
"nissan rẻ",
          "Theo mình thì không nên đăng những bài như thế này. Cái xấu xa, mình phải quên nó đi, cho nó mất dần. K nên nhắc lại. Ng tốt sẽ bị ám ảnh, k tốt cho tinh thần, ng xấu sẽ ghi nhận. Ng k hiểu biết sẽ ghi nhớ. Và nếu nhóm còn đăng nhiều bài như t… thế này thì mình sẽ rời nhóm. Cuộc sống rất ngắn ngủi, tại sao phải để tâm đến điều cần quên đi. Hãy sống tích cực và tươi sáng.",
          "Ai mượn bạn nói mà cái địt mẹ bạn lúc lồn nào cũng aI hỎi?. Là tôi, tôi đã hỏi. Thật là một cực hình khi phải nặng gánh trách nhiệm của một kẻ tội đồ đã đưa ra câu hỏi và giấu mình vào ngôi thứ ba.",
          "Có Kanavi đứng đợi. Faker lao vào đẩy được Ruler về và Oner hứng, mạng shutdown dành cho Zeus, hợp chất điên loạn thả cực kỳ đẹp và Missing cũng nằm luôn, nhưng 369 đang quẩy, anh ta cố gắng quẩy nhưng Faker đứng yên và xỉa, 369 chịu không nổi sát thương của T1, và T1 Quét Sạch JingDongggg.",
          "Hello mấy thằng trong group, tao mới vào group nên tao nói cho hiểu Tao dạo này không được vui, công việc áp lực nên tao sẽ hơi cay cú, bởi thế nên: - Thằng nào chọc tao là tao mách mẹ - Thằng nào bắt nạt tao? Tao sẽ tức và khóc tiếng của chó, sau đó khóc với mẹ",
          "ÔI TRỜI ƠI TIÊN TRI VŨ TRỤ TRẦN DẦN HÀI HƯỚC QUÁ KHOẢNH KHẮC BRUH EPIC GAMING MOMENTS XDXDXDXDXD CƯỜI TỤT CẶC CÀ KHỊA CÚ LỪA MÚA MAY QUAY CUỒNG LMAO",
          "ora ora ora muda muda đó là tao dio ủa có healthy có balance không đụ tao bắn mày á ping cao quá lag bóng long xiên nứng lồn liêm sỉ cà khịa đảng chói lóa địt mẹ wibu dừa lồn tao cười tao ỉa ngộ ha meme là ảnh chế yare yare tà tữa lofi bart simpson dexty gấu trắng trầm cảm billie eyelash drake meme anh da đen nhanh trí khi mày địt mẹ bts rác xì trây hạ đẳng team mặn uống nước nhớ nguồn team cậu team đảng team nhạt team gắt meme anime 4 ô 6 ô trung kỳ bắc kỳ nam kỳ"
]

results = classifier(texts)
for text, result in zip(texts, results):
    #print(result['label'])
    print(f"Text: {text}")
    print(f"Prediction: Label {result['label']} (Confidence: {result['score']:.4f})")
    print("-" * 50)



Text: hôm nay đen vãi lồn
Prediction: Label LABEL_1 (Confidence: 0.7916)
--------------------------------------------------
Text: cay vcl
Prediction: Label LABEL_1 (Confidence: 0.8536)
--------------------------------------------------
Text: chào cậu
Prediction: Label LABEL_0 (Confidence: 0.9937)
--------------------------------------------------
Text: ăn ba tô cơm
Prediction: Label LABEL_0 (Confidence: 0.9939)
--------------------------------------------------
Text: nissan rẻ
Prediction: Label LABEL_0 (Confidence: 0.9818)
--------------------------------------------------
Text: Theo mình thì không nên đăng những bài như thế này. Cái xấu xa, mình phải quên nó đi, cho nó mất dần. K nên nhắc lại. Ng tốt sẽ bị ám ảnh, k tốt cho tinh thần, ng xấu sẽ ghi nhận. Ng k hiểu biết sẽ ghi nhớ. Và nếu nhóm còn đăng nhiều bài như t… thế này thì mình sẽ rời nhóm. Cuộc sống rất ngắn ngủi, tại sao phải để tâm đến điều cần quên đi. Hãy sống tích cực và tươi sáng.
Prediction: Label LABEL_0 (Confidence: 0